# Application of Radial Equilibrium Equation for a Rotor
## Adaptation of the code RE-DES by Lewis (Turbomachines Performance Analysis)

The radial equilibrium equation can be reordened and written as

$$\boxed{
\frac{\text{d}}{\text{d}r} c_x(r)^2 = 2\left( \omega - \frac{c_\theta(r)}{r}\right)
\frac{\text{d} (rc_\theta(r))}{\text{d}r}
}  \tag{1}
$$

that gives the solution
$$
c_x(r) = \sqrt{f(r) + k} \tag{2}
$$
where
$$
f(r) = \int_{r_h}^r \left( \omega - \frac{c_\theta(r)}{r}\right)\text{d} (rc_\theta(r)) \tag{3}
$$

The aim is, given all the data: $Q$, $\omega$, $r_h$, $r_t$ and the function
$c_\theta(r)$, compute $c_x(r)$ and the angle $\beta_2(r)$ that fullfils the
required flowrate.

The necessary modules are imported

In [1]:
import numpy as np
import pandas as pd
pd.set_option('display.precision', 2)
from scipy.interpolate import CubicSpline
from scipy.integrate import cumulative_trapezoid, trapezoid

The data for the fan is input in this cell.
The hub to tip ratio and the rms radius, $r_\text{rms} = \sqrt{\frac{r_h^2+r_t^2}{2}}$ are computed, and the swirl distribution is also chosen.
For numerical computations, the $r$ domain is divided in $m$ intervals, and $n$ for output of results. Velocity at $r_{rms}$, $c_{\theta,\text{rms}}=c_\theta(r=r_\text{rms})$, is calculated from Euler's equation, and $c_x$ in $r_\text{rms}$ is as well estimated, assuming that it is the average velocity given by the flow rate, $\overline{c_x}$.

In [2]:
rho = 1.2                           # Density of air, kg/m³
Dh = 0.064                          # Diameter of hub, m
rh = Dh/2                           # Radius of hub, m
Dt = 0.25                           # Diameter of tip, m
rt = Dt/2                           # Radius of tip, m
h = rh/rt                           # Hub-tip ratio
rrms = np.sqrt(0.5*(rh*rh+rt*rt))   # RMS radius, m
n = 10                              # number of outputs
m = 601                             # number of interpolation points
r = np.linspace(rh,rt,m)            # Discretization of the radius for interpolation, m
Qdata = 1716                        # Flow rate, m³/h
Qdata = Qdata/3600                  # Flow rate, m³/s
omega = 2275                        # Rotational speed, rpm
omega = omega*np.pi/30              # Rotational speed, rad/s
Delta_p0_target = 75                # Pressure rise, Pa

ctrms = Delta_p0_target/(rho*omega*rrms)         # c_theta,rms, m/s
cxm = Qdata/(np.pi*(rt*rt-rh*rh))   # c_x,rms, m/s
print("Flow rate = {:0.4f} m³/s".format(Qdata))
print("Hub to tip ratio = {:0.4f}".format(h))
print("rms = {:.4f} m".format(rrms))
print("ctheta_rms = {:.4f} m/s".format(ctrms))
print("cx_rms = {:.4f} m/s".format(cxm))


Flow rate = 0.4767 m³/s
Hub to tip ratio = 0.2560
rms = 0.0912 m
ctheta_rms = 2.8754 m/s
cx_rms = 10.3916 m/s


- **Free Vortex**:
  $$ c_\theta = A/r $$

In [3]:
# User choice: "free", "forced", "constant", "mixed", "arbitrary"
flow_type = "mixed"

if flow_type == "free":
    B = ctrms*rrms
    ctheta = B/r
    print("Free vortex flow")
    print("B = {:.4f}".format(B))
elif flow_type == "forced":
    A = ctrms/(rrms)
    ctheta = A*r
    print("Forced vortex flow")
    print("A = {:.4f}".format(A))
elif flow_type == "constant":
    ctheta = ctrms*np.ones(r.size)
    print("Constant ctheta flow")
    print("ctheta = {:.4f}".format(ctrms))
elif flow_type == "mixed":
    delta_ctheta = 0.5
    B = ctrms*rrms
    A = delta_ctheta/(rt-rh) + B/(rt*rh)
    ctheta = A*(r-rrms) + B/r
    print("Mixed vortex flow")
    print("A = {:.4f}".format(A))
    print("B = {:.4f}".format(B))
elif flow_type == "arbitrary":
    ctheta = 2*omega/3*r-9.08/np.sqrt(r)

Mixed vortex flow
A = 70.9622
B = 0.2623


And now, the function $f(r)$ is computed by numerical integration (Eq. (3))

In [4]:
f = cumulative_trapezoid(omega-np.divide(ctheta,r),
                         np.multiply(r,ctheta),initial=0)

### First approximation

The first aproximation of the value of $k$ is with the assumption that
$c_{x,\text{rms}} = \overline{c_x}$

In [5]:
frms = CubicSpline(r,f)(rrms)
k = cxm*cxm-frms
print("First approximation of k: \n k = {:.4f} m²/s²".format(k))

First approximation of k: 
 k = 79.5179 m²/s²


The obtained solution is printed

In the first instance we define $D_F$ as 0.5, a conservative figure, considering that 0.6 would be a perfect performance for the profile. Also, $C_D≈0$ is estimated since the lift is expected to be maximum and the drag is expected to be minimum, so 0.01 is taken as $C_D$.

With the above assumptions and data, ${C_L}$, ${σ}$ (solidity) and ${c_x}$ along the entire length of the profile are calculated.

With these data, the chord of the profile along the length of the blade is calculated, thus defining the change of dimensions that the blade will have.


$$
σ = \frac {cos(β_1) (tan(β_1) - tan(β_2))}{2 D_F - 2  [1 - \frac{cos(β_1)}{cos(β_2) } ] } \tag{4}
$$

$$
C_L = \frac {2  cos(β_m)  (tan(β_1) - tan(β_2))}{σ} - C_Dtan(β_m) \tag{5}
$$

The contribution of Samuel Limonchi (course 2023-24 of MUREM) to this part of the notebook is acknowledged. 

In [6]:
DF_target = 0.3
CD = 0.01
Nblades = 5
def solve_fan(k):
    cx = np.sqrt(k + f)
    Q = 2*np.pi*trapezoid(np.multiply(r,cx),r)
    Delta_p0 = rho * omega * r * ctheta
    Delta_p0_avg = 2*trapezoid(np.multiply(r,Delta_p0),r)/(rt*rt-rh*rh)
    data_list = []
    #print("{:^15}{:^10}{:^10}{:^10}{:^10}{:^10}{:^10}{:^10}{:^10}{:^10}{:^10}"
    #      .format("Radius","ctheta","cx","alpha2","beta1","beta2","beta_m","Solidity","CD","CL","chord"))
    #print("\u2500"*80)
    for i in range(n):
        rdata = rh + (rt-rh)*i/(n-1)
        cthetadata = float(CubicSpline(r,ctheta)(rdata)) # That is in order to treat it as a float and not an array
        cxans = float(CubicSpline(r,cx)(rdata)) # That is in order to treat it as a float and not an array
        alpha2 = np.rad2deg(np.arctan(cthetadata/cxans))
        beta2 = np.rad2deg(np.arctan((omega*rdata-cthetadata)/cxans))
        beta1 = np.rad2deg(np.arctan(omega*rdata/cxans))
        beta_m = 0.5*(beta1+beta2)
        cosbeta1 = np.cos(np.deg2rad(beta1))
        cosbeta2 = np.cos(np.deg2rad(beta2))
        tanbeta1 = np.tan(np.deg2rad(beta1))
        tanbeta2 = np.tan(np.deg2rad(beta2))
        tanbetam = 0.5*(tanbeta1 + tanbeta2)
        beta_m = np.rad2deg(np.arctan(tanbetam))
        cosbetam = 1 / np.sqrt(1 + tanbetam*tanbetam)
        solidity = (cosbeta1* (tanbeta1 - tanbeta2)) / (2*DF_target - 2 * (1 - (cosbeta1/cosbeta2) ))
        bladespace = 2*np.pi*rdata/Nblades
        chord =  solidity * bladespace
        CL = (2/solidity) * (cosbetam * (tanbeta1 - tanbeta2)) - CD*tanbetam
        Delta_p0_data = rho * omega * rdata * cthetadata
        data_list.append({
            "Radius (m)": rdata,
            "c_theta (m/s)": cthetadata,
            "c_x (m/s)": cxans,
            "alpha_2 (deg)": alpha2,
            "beta_1 (deg)": beta1,
            "beta_2 (deg)": beta2,
            "beta_m (deg)": beta_m,
            "Solidity": solidity,
            "CL": CL,
            "Chord (mm)": chord * 1000,
            "Delta p0 (Pa)": Delta_p0_data
        })
        #print("{:^15.4f}{:^10.2f}{:^10.2f}{:^10.2f}{:^10.2f}{:^10.2f}{:^10.2f}{:^10.2f}{:^10.2f}{:^10.2f}{:^10.2f}".
        #        format(rdata,cthetadata,cxans,alpha2,beta1,beta2,beta_m,solidity,CD,CL,chord))
    print("Q = {:.3f} m^3/s".format(Q))
    errorQ = np.abs(Qdata-Q)/Qdata * 100
    print("error in flow rate = {:.1e} %".format(errorQ))
    print("Delta p0 avg = {:.2f} Pa".format(Delta_p0_avg))
    errorP0 = np.abs(Delta_p0_target-Delta_p0_avg)/Delta_p0_target * 100
    print("error in pressure rise = {:.1e} %".format(errorP0))
    df = pd.DataFrame(data_list)
    df = df.round({"Chord (mm)":0})
    return df

df = solve_fan(k)

Q = 0.483 m^3/s
error in flow rate = 1.3e+00 %
Delta p0 avg = 81.58 Pa
error in pressure rise = 8.8e+00 %


In [7]:
df

,Radius (m),c_theta (m/s),c_x (m/s),alpha_2 (deg),beta_1 (deg),beta_2 (deg),beta_m (deg),Solidity,CL,Chord (mm),Delta p0 (Pa)
0,0.03,3.99,8.92,24.13,40.53,22.14,32.25,1.41,0.53,57.0,36.54
1,0.04,2.73,8.82,17.18,48.84,39.85,44.69,0.65,0.67,34.0,33.00
2,0.05,2.24,8.85,14.23,54.81,49.34,52.24,0.40,0.77,26.0,33.79
3,0.06,2.16,9.05,13.43,58.91,54.84,56.99,0.31,0.81,25.0,38.91
4,0.07,2.31,9.42,13.76,61.67,58.16,60.01,0.29,0.82,27.0,48.36
5,0.08,2.60,9.93,14.66,63.51,60.18,61.94,0.30,0.81,31.0,62.15
6,0.09,2.99,10.57,15.77,64.73,61.42,63.17,0.31,0.79,37.0,80.27
7,0.10,3.44,11.31,16.93,65.53,62.15,63.94,0.34,0.77,44.0,102.72
8,0.11,3.95,12.13,18.04,66.06,62.57,64.42,0.37,0.75,53.0,129.50
9,0.12,4.49,13.01,19.06,66.40,62.78,64.71,0.40,0.73,62.0,160.61


### More precise computation


In [8]:
from scipy.optimize import brentq

In [9]:
def QFunction(k):
    cx = np.sqrt(k + f)
    Q_temptative = 2*np.pi*trapezoid(np.multiply(r,cx),r)
    return Qdata-Q_temptative

k = brentq(QFunction,0.5*k,1.5*k)
print("k = {:.4f} m²/s²".format(k))

k = 76.7008 m²/s²


In [10]:
df = solve_fan(k)
df

Q = 0.477 m^3/s
error in flow rate = 8.2e-14 %
Delta p0 avg = 81.58 Pa
error in pressure rise = 8.8e+00 %


,Radius (m),c_theta (m/s),c_x (m/s),alpha_2 (deg),beta_1 (deg),beta_2 (deg),beta_m (deg),Solidity,CL,Chord (mm),Delta p0 (Pa)
0,0.03,3.99,8.76,24.52,41.04,22.51,32.72,1.48,0.51,59.0,36.54
1,0.04,2.73,8.66,17.48,49.36,40.37,45.22,0.66,0.66,35.0,33.00
2,0.05,2.24,8.69,14.48,55.30,49.86,52.75,0.40,0.77,27.0,33.79
3,0.06,2.16,8.89,13.66,59.35,55.31,57.45,0.32,0.81,25.0,38.91
4,0.07,2.31,9.27,13.98,62.06,58.57,60.41,0.29,0.82,27.0,48.36
5,0.08,2.60,9.79,14.86,63.84,60.54,62.28,0.30,0.81,31.0,62.15
6,0.09,2.99,10.44,15.97,65.01,61.72,63.46,0.31,0.79,37.0,80.27
7,0.10,3.44,11.19,17.11,65.77,62.42,64.19,0.34,0.77,44.0,102.72
8,0.11,3.95,12.01,18.20,66.26,62.79,64.64,0.37,0.75,53.0,129.50
9,0.12,4.49,12.90,19.21,66.58,62.97,64.90,0.40,0.72,62.0,160.61
